In [ ]:
import sys
# sys.path.append("/kaggle/working/DL-BHW-2/src")
from train_routine import create_dataloaders_tf
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchtext
from transformer_model import Transformer
from torch.utils.data import DataLoader
from train_routine import train_tf, evaluate_b


torchtext.disable_torchtext_deprecation_warning()


In [ ]:
# PATH_TO_DATA = "/kaggle/working/DL-BHW-2/data"
PATH_TO_DATA = "../data"

In [ ]:
device = torch.device("cuda")
train_loader, val_loader, test_loader = create_dataloaders_tf(path_to_data=PATH_TO_DATA, batch_size=32, device=device)


model = Transformer(
    src_vocab_size=train_loader.dataset.de.vocab_size,
    tgt_vocab_size=train_loader.dataset.en.vocab_size,
    src_seq_len=80,
    tgt_seq_len=80,
    d_model=256,
    N=3,
    h=8,
    dropout=0.1,
    d_ff=512
)

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,epochs=10, steps_per_epoch=len(train_loader)
)
criterion = nn.CrossEntropyLoss(ignore_index=1, label_smoothing=0.1)

train_tf(model, train_loader, val_loader, optimizer, criterion, scheduler, device, n_epochs=10)

Epoch 1 / 10 : train x-entropy : 6.070421964701219, validation BLEU : 6.620839507717983
Epoch 2 / 10 : train x-entropy : 4.579386408204942, validation BLEU : 21.47475963533351


KeyboardInterrupt: 

In [ ]:
with torch.no_grad():
    test_preds = evaluate_b(model, test_loader, train_loader.dataset.en, test=True, use_beam_search=True)

In [ ]:
with open("./predictions.txt", "w") as f:
    test_preds_for_bleu = list(map(lambda x : ' '.join(x), test_preds))
    for hyp in test_preds_for_bleu:
        print(hyp, file=f)


torch.save(model.state_dict(), "./transformer.pth")

import wandb
# wandb.login(key=wandb_api_key)
run = wandb.init(
    entity="chagrygoris",
    project="DL-BHW-2"
)



artifact = wandb.Artifact(name="predictions.txt", type="predictions")
artifact.add_file(local_path="./predictions.txt", name="predictions.txt")
artifact.add_file(local_path="./transformer.pth", name="transformer.pth")
run.log_artifact(artifact)


run.finish()